In [ ]:
#Installing basic dependencies in Colab
!pip install --upgrade pip
!pip install torch transformers sentence-transformers faiss-cpu numpy PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 9.6 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 39.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [faiss-cpu]


In [ ]:
# Install required packages (run once per runtime)
!pip install sentence-transformers faiss-cpu numpy piper-tts soundfile PyPDF2 python-docx pyngrok fastapi nest_asyncio uvicorn --quiet

# Standard Python imports
import os
import io
import re
import numpy as np
import faiss
from google.colab import files
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import piper
import soundfile as sf


In [ ]:
# Loading QA model (for answering once we give it context)
qa_pipeline = pipeline(
    "question-answering",
    model="distilbert-base-cased-distilled-squad",
    tokenizer="distilbert-base-cased",
    device=-1,  # CPU‑friendly
)

print("✅ QA pipeline loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ QA pipeline loaded.


In [ ]:
# Embedding model for RAG
embedding_model = SentenceTransformer("sentence-transformers/multi-qa-MiniLM-L6-cos-v1")

# Semantic chunking function (no reliance on 'text' yet)
def sentence_chunk_text(text, max_chars=250):
    paragraphs = re.split(r"\n\s*\n", text.strip())
    chunks = []
    for para in paragraphs:
        para = para.strip()
        if not para:
            continue
        sentences = re.split(r"(?<=[.!?])\s+", para)
        current = ""
        for sent in sentences:
            sent = sent.strip()
            if not sent:
                continue
            if len(current) + len(sent) + 1 <= max_chars:
                current = current + " " + sent if current else sent
            else:
                if current:
                    chunks.append(current.strip())
                current = sent
        if current:
            chunks.append(current.strip())
    return chunks

print("✅ Embedding model and chunking helper ready.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model and chunking helper ready.


In [ ]:
# TTS model setup
MODEL_DIR = "/content/piper_models"
os.makedirs(MODEL_DIR, exist_ok=True)

model_name = "en_US-lessac-medium.onnx"
model_path = os.path.join(MODEL_DIR, model_name)

if not os.path.exists(model_path):
    print("Downloading Piper model...")
    model_url = "https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/en/en_US/lessac/medium/en_US-lessac-medium.onnx"
    import requests
    r = requests.get(model_url)
    r.raise_for_status()
    with open(model_path, "wb") as f:
        f.write(r.content)
    print("Downloaded to", model_path)

config_name = model_name + ".json"
config_path = os.path.join(MODEL_DIR, config_name)

if not os.path.exists(config_path):
    print("Downloading config...")
    config_url = "https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/en/en_US/lessac/medium/en_US-lessac-medium.onnx.json"
    r = requests.get(config_url)
    r.raise_for_status()
    with open(config_path, "w") as f:
        f.write(r.text)
    print("Config downloaded.")

# Load the voice model
voice_model = piper.PiperVoice.load(model_path)

print("✅ TTS voice model loaded.")

Downloaded to /content/piper_models/en_US-lessac-medium.onnx
Config downloaded.
✅ TTS voice model loaded.


In [ ]:
# Shareable state for FastAPI endpoints
global current_index, current_document, DOCUMENTS

current_index = None
current_document = "No document uploaded yet."
DOCUMENTS = []
print("✅ Global RAG / upload state initialized.")

✅ Global RAG / upload state initialized.


In [ ]:
!pip install pyngrok

In [ ]:
!pip install fastapi nest_asyncio uvicorn

import base64
import nest_asyncio
import uvicorn
from fastapi import FastAPI, File, UploadFile
from fastapi.responses import HTMLResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pyngrok import ngrok, conf

# Authenticate ngrok
conf.get_default().auth_token = "3BeByzmCcpNwwJetNz2HtYXYNxQ_4NHpq8s8djMtviopanp3f"

# FastAPI app
app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# Data model
class AskRequest(BaseModel):
    question: str

#1. FILE UPLOAD: rebuild index from user file

@app.post("/upload")
async def upload_file(file: UploadFile = File(...)):
  try:
        global current_document, DOCUMENTS, current_index
        # Read file bytes
        content = await file.read()
        filename = file.filename.lower()
        text = ""

        if filename.endswith(".docx"):
            from docx import Document
            doc = Document(io.BytesIO(content))
            text = "\n".join(p.text for p in doc.paragraphs)
        elif filename.endswith(".pdf"):
            from PyPDF2 import PdfReader
            reader = PdfReader(io.BytesIO(content))
            text = ""
            for page in reader.pages:
                text += (page.extract_text() or "") + "\n"
        else:
            text = content.decode("utf‑8")

        current_document = text

        # Use the chunking function from earlier
        DOCUMENTS = sentence_chunk_text(current_document, max_chars=250)

        # Rebuild embeddings and index
        embeddings = embedding_model.encode(
            DOCUMENTS,
            normalize_embeddings=True,
        )
        dimension = embeddings.shape[1]
        current_index = faiss.IndexFlatIP(dimension)
        current_index.add(np.array(embeddings, dtype=np.float32))

        return JSONResponse(
            {"message": f"Uploaded and indexed {len(DOCUMENTS)} chunks from {file.filename}"}
        )
  except Exception as e:
        print("Error in upload:", e)
        return JSONResponse({"error": str(e)})
#2. SINGLE CHUNK RETRIEVAL

def retrieve_single_chunk(query, k=1):
    query_emb = embedding_model.encode([query])
    distances, indices = current_index.search(query_emb.astype(np.float32), k)
    i = indices[0][0]
    score = distances[0][0]
    return DOCUMENTS[i], score

#3. TTS helper (reuseable)

def answer_to_audio_base64(answer_text: str) -> str:
    audio_frames = list(voice_model.synthesize(answer_text))
    wav = np.concatenate(
        [f.audio_float_array for f in audio_frames],
        axis=0,
    )
    if wav.ndim == 1:
        wav = wav.reshape(-1, 1)
    buf = io.BytesIO()
    sf.write(buf, wav, samplerate=22050, format="WAV")
    buf.seek(0)
    audio_b64 = base64.b64encode(buf.read()).decode("utf-8")
    return audio_b64

#4. QA endpoint: /ask

@app.post("/ask")
async def ask_api(request: AskRequest):
    if current_index is None or len(DOCUMENTS) == 0:
        return {"error": "No document indexed. Please upload a file first."}

    try:
        single_chunk, score = retrieve_single_chunk(request.question, k=1)
        if not single_chunk:
            return {"answer_text": "No context found.", "audio": None}

        result = qa_pipeline(question=request.question, context=single_chunk)
        answer = result["answer"]

        audio_b64 = answer_to_audio_base64(answer)

        return {
            "question": request.question,
            "answer_text": answer,
            "audio": audio_b64,
            "sample_rate": 22050,
            "score": score,
        }
    except Exception as e:
        return {"error": str(e)}

#5. HTML/JS Frontend (with upload button)

frontend_html = """
<!DOCTYPE html>
<html>
<head>
  <meta charset="UTF-8">
  <title>RAG + TTS with Upload</title>
  <style>
    body { font-family:Arial; margin:40px; }
    .section { margin-bottom:20px; }
    input, button { padding:10px; font-size:1rem; }
    audio { margin-top:10px; width:100%; }
  </style>
</head>
<body>

  <h2>Upload your document</h2>
  <div class="section">
    <input type="file" id="docUpload" accept=".pdf,.docx,.txt">
    <button onclick="uploadDoc()">Upload</button>
    <div id="uploadStatus"></div>
  </div>

  <h2>Ask the Document Assistant</h2>
  <div class="section">
    <input id="questionInput" type="text" style="width:70%" placeholder="Ask a question...">
    <button onclick="askQuestion()">Ask</button>
    <div id="answerBox"></div>
  </div>

  <script>
    const API_URL = "https://ashlie-unconniving-autumn.ngrok-free.dev";

    async function uploadDoc() {
      const input = document.getElementById("docUpload");
      const file = input.files[0];
      if (!file) return alert("Please select a file.");

      const fd = new FormData();
      fd.append("file", file);

      const status = document.getElementById("uploadStatus");
      status.innerText = "Uploading...";

      try {
        const r = await fetch(`${API_URL}/upload`, {
          method: "POST",
          body: fd,
        });

        const data = await r.json();

        if (r.ok) {
          status.innerHTML = `<b>Uploaded:</b> ${data.message}`;
        } else {
          status.innerHTML = `<b>Error:</b> ${data.error || 'Unknown error'}`;
        }
      } catch (err) {
        console.error("Upload error:", err);
        status.innerHTML = `<b>Error:</b> ${err.message}`;
      }
    }

    async function askQuestion() {
      const question = document.getElementById("questionInput").value.trim();
      if (!question) return alert("Enter a question.");

      const r = await fetch(`${API_URL}/ask`, {
        method: "POST",
        headers: { "Content-Type": "application/json" },
        body: JSON.stringify({ question }),
      });

      const data = await r.json();
      if (data.error) {
        document.getElementById("answerBox").innerHTML = `<b>Error:</b> ${data.error}`;
        return;
      }

      const audioSrc = `data:audio/wav;base64,${data.audio}`;
      const html = `
        <b>Answer:</b> ${data.answer_text}
        <br><audio controls autoplay src="${audioSrc}"></audio>
        <p><small>Chunk relevance score: ${data.score.toFixed(3)}</small></p>
      `;
      document.getElementById("answerBox").innerHTML = html;
    }
  </script>
</body>
</html>
"""

@app.get("/", response_class=HTMLResponse)
async def root():
    return HTMLResponse(frontend_html)

#6. Start ngrok + server

tunnel = ngrok.connect(8000)
print("🌐 Public URL ->", tunnel.public_url)

nest_asyncio.apply()
config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="error")
server = uvicorn.Server(config)
await server.serve()

PyngrokNgrokInstallError: An error occurred while downloading ngrok from https://bin.ngrok.com/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.zip: HTTP Error 403: Forbidden